In [ ]:
import torch
from torch import logspace

from occhio import ToyModel
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions import SparseUniform
from occhio.model_grid import ModelGrid, Axis
from occhio.visualization_2 import EmbeddingPlot, ReconstructionLossPlot

In [ ]:
DEVICE = "mps"

In [ ]:
N_FEATURES = 400


def create_model(params) -> ToyModel:
    gen = torch.Generator(DEVICE).manual_seed(23)
    return ToyModel(
        distribution=SparseUniform(
            N_FEATURES,
            [1 / (i + 1) for i in range(N_FEATURES)],
            generator=gen,
        ),
        ae=TiedLinearRelu(N_FEATURES, int(params["N_HIDDEN"]), generator=gen),
        device=DEVICE,
    )


grid = ModelGrid(
    create_model,
    [
        Axis("N_HIDDEN", values=[10, 20, 30, 40, 50, 60, 70]),
    ],
)
history_grid = grid.fit(15000, snapshot_interval=200)

In [ ]:
import importlib

import occhio.visualization_2.plots.reconstruction_loss as rl

importlib.reload(rl)

reconstruction_loss_plot = rl.ReconstructionLossPlot()
reconstruction_loss_plot(history_grid, render_axes=("Epoch",))

In [ ]:
N_FEATURES = 400


def create_model(params) -> ToyModel:
    gen = torch.Generator(DEVICE).manual_seed(23)
    return ToyModel(
        distribution=SparseUniform(
            N_FEATURES,
            1 - params["Sparsity"],
            generator=gen,
        ),
        importances=0.99 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, int(params["N_HIDDEN"]), generator=gen),
        device=DEVICE,
    )


grid = ModelGrid(
    create_model,
    [
        Axis("N_HIDDEN", values=[10, 20, 30, 40, 50, 60, 70]),
        Axis(label="Sparsity", values=logspace(-2, 0, 5)),
    ],
)
history_grid = grid[:, -2:].fit(25000, snapshot_interval=200)

In [ ]:
import importlib

import occhio.visualization_2.plots.reconstruction_loss as rl

importlib.reload(rl)

reconstruction_loss_plot = rl.ReconstructionLossPlot()
reconstruction_loss_plot(history_grid, render_axes=("Epoch",), height=1200)